Model Setup

In [ ]:
import jax.numpy as jnp  # JAX's numpy-compatible array API, used for all numerical arrays
import numpyro  # probabilistic programming library built on JAX
import numpyro.distributions as dist  # distribution classes (Categorical, Dirichlet, etc.)

import dynestyx as dsx  # project package providing dynamical-model sampling utilities
from dynestyx import DynamicalModel  # container class describing a state-space model's dynamics


def hmm_model(A=None, obs_times=None, obs_values=None, predict_times=None):
    # Define the 2x2 transition matrix.
    # Each row is a Dirichlet(1,1) draw so it sums to 1; `.expand([2])` gives one row per state,
    # and `.to_event(1)` treats each row's 2 entries as a single event (not independent dims).
    A = numpyro.sample("A", dist.Dirichlet(jnp.ones(2)).expand([2]).to_event(1), obs=A)

    def state_evolution(x, u, t_now, t_next):
        # Given current state x, the next state is Categorical with transition
        # probabilities taken from row x of the transition matrix A.
        return dist.Categorical(probs=A[x])

    def observation_model(x, u, t):
        # Emission distribution over 6 possible observations, indexed by hidden state x.
        # State 0 emits each of 6 outcomes uniformly (a "fair die").
        # State 1 is biased: outcomes 0-4 are equally likely but outcome 5 has probability 1/2
        # (a "loaded die").
        probs = jnp.array(
            [
                [1 / 6, 1 / 6, 1 / 6, 1 / 6, 1 / 6, 1 / 6],
                [1 / 10, 1 / 10, 1 / 10, 1 / 10, 1 / 10, 1 / 2],
            ]
        )
        return dist.Categorical(probs=probs[x])

    # Bundle the initial-state distribution, transition (state_evolution), and
    # emission (observation_model) functions into a single DynamicalModel spec.
    dynamics = DynamicalModel(
        initial_condition=dist.Categorical(probs=jnp.ones(2) / 2),  # start state is 50/50
        state_evolution=state_evolution,
        observation_model=observation_model,
    )

    # Draw (or condition on) a trajectory named "f" from the dynamics, optionally
    # conditioning on observed data (obs_times/obs_values) and/or requesting
    # predictions at additional predict_times.
    return dsx.sample(
        "f",
        dynamics,
        obs_times=obs_times,
        obs_values=obs_values,
        predict_times=predict_times,
    )

Generating data (same logic)

In [ ]:
import jax.random as jr  # JAX's explicit PRNG key API
from numpyro.infer import Predictive  # draws samples by running a model forward (prior/predictive)

from dynestyx import DiscreteTimeSimulator, flatten_draws  # simulator context + helper (unused here)

n_rollout_eval = 100  # number of extra timesteps held out for rollout evaluation
n_train = 10000  # number of timesteps used for training/inference
# Full time grid spans training period plus the held-out rollout-evaluation period.
obs_times_full = jnp.arange(start=0.0, stop=n_train + n_rollout_eval, step=1.0)
obs_times = obs_times_full[:n_train]  # times used for fitting the model
rollout_eval_times = obs_times_full[n_train:]  # times reserved for evaluating rollouts

prng_key = jr.PRNGKey(0)  # fixed random seed for reproducibility
# Predictive wraps hmm_model to draw 1 sample of the full generative process (prior predictive).
predictive_model = Predictive(hmm_model, num_samples=1)

# Ground-truth transition matrix used to simulate the synthetic dataset.
true_A = jnp.array([[0.95, 0.05], [0.1, 0.9]])

# DiscreteTimeSimulator context tells dsx.sample to actually simulate a trajectory
# forward in discrete time steps (rather than, e.g., run inference).
with DiscreteTimeSimulator():
    synthetic_samples = predictive_model(prng_key, A=true_A, predict_times=obs_times_full)

# Extract observations from synthetic data.
# f_states / f_observations have shape (num_samples, n_sim, T, 1) — the trailing
# singleton is the state_dim convention used for all simulators.  Index [0, 0, :, 0]
# to recover a 1-D array suitable for plotting and conditioning.
print(
    "synthetic shapes:",
    synthetic_samples["f_times"].shape,
    synthetic_samples["f_states"].shape,
    synthetic_samples["f_observations"].shape,
)
obs_all = synthetic_samples["f_observations"][0, 0, :, 0]  # flatten to 1-D array of observed dice rolls
states_all = synthetic_samples["f_states"][0, 0, :, 0]  # flatten to 1-D array of true hidden states
obs_values = obs_all[:n_train]  # training observations
obs_values_eval_rollout = obs_all[n_train:]  # held-out observations for rollout evaluation

states_true = states_all[:n_train]  # true hidden states for the training period
states_true_eval_rollout = states_all[n_train:]  # true hidden states for the rollout-evaluation period


There is an utility dynestyx provides that allows us to plot HMM observations

In [ ]:
from dynestyx.diagnostics.plotting_utils import plot_hmm_states_and_observations  # HMM-specific plot helper

# Plot the first 100 timesteps: hidden state trajectory alongside the observed emissions,
# so we can visually confirm the synthetic data looks like a sensible HMM.
plot_hmm_states_and_observations(
    times=obs_times[:100],
    x=states_true[:100],
    y=obs_values[:100],
)

Code for Bayesian inference on HMM

In [ ]:
from numpyro.infer import MCMC, NUTS  # No-U-Turn Sampler + MCMC driver for Bayesian inference
from dynestyx import Filter  # context manager selecting an inference/filtering backend
from dynestyx.inference.filters import HMMConfig  # config telling Filter to use exact HMM forward-filtering

mcmc_key = jr.PRNGKey(0)  # separate PRNG key for the MCMC run

# Inside this context, dsx.sample marginalizes out the discrete hidden states exactly
# using the HMM forward algorithm (instead of, e.g., sampling them discretely),
# which lets NUTS (a gradient-based sampler) work on the continuous parameter A.
with Filter(filter_config=HMMConfig()):
    nuts_kernel = NUTS(hmm_model)  # NUTS kernel targeting the (marginalized) hmm_model posterior
    mcmc = MCMC(
        nuts_kernel,
        num_samples=500,  # number of posterior samples to collect
        num_warmup=500,  # number of warmup/adaptation steps (discarded)
    )
    # Run MCMC conditioning on the training observations; A is left unspecified so it's inferred.
    mcmc.run(mcmc_key, obs_times=obs_times, obs_values=obs_values)

posterior_samples = mcmc.get_samples()  # dict of posterior draws, e.g. posterior_samples["A"]

import matplotlib.pyplot as plt  # plotting
import seaborn as sns  # plot styling (used here just for despine)

# Histogram of posterior draws for the 0->1 transition probability (A[0,1]).
plt.hist(
    posterior_samples["A"][:, 0, 1],
    bins=20,
    color="k",
    alpha=0.5,
    label=r"p0→1",
)
plt.axvline(true_A[0, 1], color="k", linestyle="--")  # mark the true value used to simulate the data
# Histogram of posterior draws for the 1->0 transition probability (A[1,0]).
plt.hist(
    posterior_samples["A"][:, 1, 0],
    bins=20,
    color="b",
    alpha=0.5,
    label=r"p1→0",
)
plt.axvline(true_A[1, 0], color="b", linestyle="--")  # mark the true value used to simulate the data
sns.despine()  # remove top/right plot spines for a cleaner look
plt.legend()
plt.show()

We can evaluate filter + simulator similarly